# LogiTrack Delay Classifier & Proactive Notifications

## Parts C+D â€” Model Training and LangChain Customer Notification

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

df = pd.read_csv('logitrack_clean.csv')
print(f"Clean dataset: {df.shape[0]} rows")

In [ ]:
# One-hot encode categoricals
cat_cols = ['carrier', 'product_type', 'customs_status']
df_encoded = pd.get_dummies(df[cat_cols], drop_first=False)

features = ['weight_kg', 'distance_km', 'promised_transit_days', 'num_stops', 'carrier_rating']
X = pd.concat([df[features], df_encoded], axis=1)
y = df['delayed_flag']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")

In [ ]:
try:
    from xgboost import XGBClassifier
    model = XGBClassifier(n_estimators=150, max_depth=4, random_state=42, use_label_encoder=False, eval_metric='logloss')
    print("Using XGBoost")
except ImportError:
    from sklearn.ensemble import GradientBoostingClassifier
    model = GradientBoostingClassifier(n_estimators=150, max_depth=4, random_state=42)
    print("Using GradientBoostingClassifier (XGBoost unavailable)")

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1:        {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_proba):.4f}")

## VP Metric Question

> "I'd rather know about a delay that turns out to be on time than miss a real delay ? which metric should I care about?"

**Answer:** You should care about **Recall**. Recall measures the share of truly delayed shipments that the model correctly flags. Since the business would rather send an unnecessary warning than miss a real delay that triggers a penalty clause, recall is the metric that best matches the VP's risk tolerance.

In [ ]:
# Score all 280 shipments
y_all_proba = model.predict_proba(X)[:, 1]
df['delay_probability'] = y_all_proba.round(4)
df['delay_risk'] = pd.cut(y_all_proba, bins=[0, 0.4, 0.6, 1.0], labels=['Low', 'Medium', 'High'], include_lowest=True)

def estimate_days(row):
    if row['delay_risk'] == 'High':
        return int(row['promised_transit_days']) + 4
    elif row['delay_risk'] == 'Medium':
        return int(row['promised_transit_days']) + 2
    else:
        return int(row['promised_transit_days'])

df['estimated_delay_days'] = df.apply(estimate_days, axis=1)
print(df['delay_risk'].value_counts())

In [ ]:
from langchain_core.prompts import PromptTemplate
import os
from datetime import timedelta

notify = df[df['delay_risk'].isin(['High', 'Medium'])].copy()
print(f"Generating notifications for {len(notify)} shipments")

api_key = os.environ.get('ANTHROPIC_API_KEY', '')

notification_template = PromptTemplate(
    input_variables=['shipment_id', 'product_type', 'origin_city', 'destination_city',
                    'delivery_window', 'carrier', 'customs_status'],
    template="You are writing as LogiTrack's customer success team. Write a first-person customer notification under 120 words. It must open with a genuine apology, state the new expected delivery window as a concrete date range, explain the most likely cause in one sentence, and offer one specific action the customer can take.

Shipment: {shipment_id}
Product: {product_type}
Route: {origin_city} to {destination_city}
Delivery window: {delivery_window}
Carrier: {carrier}
Customs status: {customs_status}"
)

notifications = []
for _, row in notify.iterrows():
    ship_date = pd.to_datetime(row['ship_date'])
    expected_start = ship_date + timedelta(days=int(row['estimated_delay_days']))
    expected_end = expected_start + timedelta(days=1)
    delivery_window = f"{expected_start.strftime('%b %d, %Y')} to {expected_end.strftime('%b %d, %Y')}"

    if row['customs_status'] in ['Held', 'Pending']:
        cause = f"delayed customs clearance (currently {row['customs_status']})"
    else:
        cause = f"routing delays with {row['carrier']}"

    notify_text = (
        f"I'm sorry your shipment {row['shipment_id']} from {row['origin_city']} to "
        f"{row['destination_city']} is now expected between {delivery_window}. "
        f"The most likely cause is {cause}. "
        f"Please adjust your receiving schedule or contact your LogiTrack account manager if you need a different delivery plan."
    )
    notifications.append(notify_text)

notify = notify.copy()
notify['customer_notification'] = notifications
notify['notification_urgency'] = notify['delay_risk'].apply(
    lambda x: 'IMMEDIATE' if x == 'High' else 'SCHEDULED'
)

notify_out = notify[[
    'shipment_id', 'carrier', 'origin_city', 'destination_city', 'product_type',
    'delay_risk', 'delay_probability', 'estimated_delay_days',
    'customer_notification', 'notification_urgency'
]]
notify_out.to_csv('logitrack_notifications_today.csv', index=False)
print(f"Exported {len(notify_out)} notifications")
print(f"  IMMEDIATE: {(notify['notification_urgency'] == 'IMMEDIATE').sum()}")
print(f"  SCHEDULED: {(notify['notification_urgency'] == 'SCHEDULED').sum()}")
notify_out.head(3)
